# Visual Guardian V2 — Seizure Dataset Verification + Preprocessing
**Accelerator:** GPU T4 (required — YOLO11n inference)  
**Input:** Upload `datasets/unusual_movement/data/` folder as a Kaggle dataset  
**Expected structure after upload:**
```
/kaggle/input/<your-dataset>/data/Seizure/   (403 clips)
/kaggle/input/<your-dataset>/data/Normal/    (403 clips)
```
**Output:** `seizure_preprocessed/` → save as Kaggle dataset → input for Phase 3 training notebook  
**Expected runtime:** 1–2 hours on T4

In [1]:
!pip install ultralytics

import re
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from ultralytics import YOLO

import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 24.7 MB/s eta 0:00:00a 0:00:01
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
CUDA available: True
GPU: Tesla T4


In [3]:
# ── Config ────────────────────────────────────────────────────────────────────
OUTPUT_DIR   = Path("/kaggle/working/seizure_preprocessed")
CROP_SIZE    = 224
PADDING      = 0.20
MIN_FRAMES   = 64    # seizure model: 32 frames × stride 2
MIN_DURATION = 2.0
SPLIT_RATIO  = {"train": 0.70, "val": 0.15, "test": 0.15}

# Auto-detect dataset root — find the folder containing Seizure/ and Normal/
DATASET_ROOT = "/kaggle/input/datasets/i221192muhammadhasan/seizure-data/data"


if DATASET_ROOT is None:
    raise RuntimeError(
        "Could not find Seizure/ and Normal/ folders.\n"
        "Upload the 'data' folder from datasets/unusual_movement/data as a Kaggle dataset."
    )

SEIZURE_DIR = Path("/kaggle/input/datasets/i221192muhammadhasan/seizure-data/data/Seizure")
NORMAL_DIR  = Path("/kaggle/input/datasets/i221192muhammadhasan/seizure-data/data/Normal")
print(f"Dataset root : {DATASET_ROOT}")
print(f"Seizure clips: {len(list(SEIZURE_DIR.glob('*.mp4')))}")
print(f"Normal clips : {len(list(NORMAL_DIR.glob('*.mp4')))}")

Dataset root : /kaggle/input/datasets/i221192muhammadhasan/seizure-data/data
Seizure clips: 403
Normal clips : 403


In [4]:
# ── Verify Dataset ────────────────────────────────────────────────────────────
# Filename format: S{subject}_{session}_{clip}.mp4  e.g. S47_13_175.mp4
def extract_subject_id(filename: str) -> str | None:
    m = re.match(r'S(\d+)_', filename)
    return f"sub_{int(m.group(1)):03d}" if m else None

records = []
for label, folder in [("seizure", SEIZURE_DIR), ("normal", NORMAL_DIR)]:
    for path in sorted(folder.glob("*.mp4")):
        sub_id = extract_subject_id(path.name)
        cap    = cv2.VideoCapture(str(path))
        fc     = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        fps    = cap.get(cv2.CAP_PROP_FPS)
        w      = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h      = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        cap.release()
        dur = (fc / fps) if fps > 0 else 0.0
        records.append({
            "path":        str(path),
            "filename":    path.name,
            "label":       label,
            "subject_id":  sub_id,
            "frame_count": fc,
            "fps":         round(fps, 2),
            "width":       w,
            "height":      h,
            "duration_s":  round(dur, 2),
            "valid":       fc >= MIN_FRAMES and dur >= MIN_DURATION,
        })

df = pd.DataFrame(records)
valid_df = df[df["valid"]].copy()

print(f"Total clips       : {len(df)}")
print(f"Valid (>={MIN_FRAMES} frames, >={MIN_DURATION}s) : {len(valid_df)}")
print(f"Filtered out      : {len(df) - len(valid_df)}")
print(f"\nValid by class:")
print(valid_df["label"].value_counts().to_string())
print(f"\nUnique subjects   : {valid_df['subject_id'].nunique()}")
print(f"\nFrame count stats:")
print(valid_df["frame_count"].describe().round(1).to_string())
print(f"\nDuration stats:")
print(valid_df["duration_s"].describe().round(2).to_string())

Total clips       : 806
Valid (>=64 frames, >=2.0s) : 806
Filtered out      : 0

Valid by class:
label
seizure    403
normal     403

Unique subjects   : 19

Frame count stats:
count    806.0
mean     127.8
std        1.6
min      125.0
25%      127.0
50%      127.0
75%      129.0
max      131.0

Duration stats:
count    806.00
mean       5.11
std        0.06
min        5.00
25%        5.08
50%        5.08
75%        5.16
max        5.24


In [5]:
# ── Subject-Level Split ───────────────────────────────────────────────────────
subjects = sorted(valid_df["subject_id"].dropna().unique())
np.random.seed(42)
np.random.shuffle(subjects)

n       = len(subjects)
n_train = int(n * SPLIT_RATIO["train"])
n_val   = int(n * SPLIT_RATIO["val"])

train_subs = set(subjects[:n_train])
val_subs   = set(subjects[n_train:n_train + n_val])
test_subs  = set(subjects[n_train + n_val:])

def assign_split(row):
    s = row["subject_id"]
    if s in train_subs: return "train"
    if s in val_subs:   return "val"
    return "test"

valid_df["split"] = valid_df.apply(assign_split, axis=1)

print(f"Total subjects: {n}")
print(f"  Train: {len(train_subs)} subjects, {(valid_df['split']=='train').sum()} clips")
print(f"  Val  : {len(val_subs)} subjects, {(valid_df['split']=='val').sum()} clips")
print(f"  Test : {len(test_subs)} subjects, {(valid_df['split']=='test').sum()} clips")
print(f"\nClass balance per split:")
print(valid_df.groupby(["split", "label"]).size().unstack(fill_value=0).to_string())

for s1, s2 in [("train","val"),("train","test"),("val","test")]:
    overlap = set(valid_df[valid_df["split"]==s1]["subject_id"]) & set(valid_df[valid_df["split"]==s2]["subject_id"])
    assert len(overlap) == 0, f"Subject overlap between {s1} and {s2}: {overlap}"
print("\n[OK] No subject appears in more than one split")

Total subjects: 19
  Train: 13 subjects, 572 clips
  Val  : 2 subjects, 48 clips
  Test : 4 subjects, 186 clips

Class balance per split:
label  normal  seizure
split                 
test       93       93
train     286      286
val        24       24

[OK] No subject appears in more than one split


In [6]:
# ── YOLO11n Preprocessing ─────────────────────────────────────────────────────
print("Loading YOLO11n...")
model = YOLO("yolo11n.pt")

def get_crop(frame: np.ndarray) -> tuple[np.ndarray, bool]:
    """YOLO11n person crop with fallback to resized full frame."""
    results = model(frame, verbose=False, classes=[0])
    boxes   = results[0].boxes
    if boxes is not None and len(boxes) > 0:
        best = boxes[boxes.conf.argmax()]
        x1, y1, x2, y2 = best.xyxy[0].cpu().numpy().astype(int)
        h, w  = frame.shape[:2]
        pad_x = int((x2 - x1) * PADDING)
        pad_y = int((y2 - y1) * PADDING)
        x1 = max(0, x1 - pad_x); y1 = max(0, y1 - pad_y)
        x2 = min(w, x2 + pad_x); y2 = min(h, y2 + pad_y)
        crop = frame[y1:y2, x1:x2]
        if crop.size > 0:
            return cv2.resize(crop, (CROP_SIZE, CROP_SIZE)), True
    return cv2.resize(frame, (CROP_SIZE, CROP_SIZE)), False


total, processed, no_person = len(valid_df), 0, 0

for _, row in valid_df.iterrows():
    out_dir = OUTPUT_DIR / row["split"] / row["label"] / Path(row["filename"]).stem
    out_dir.mkdir(parents=True, exist_ok=True)

    cap       = cv2.VideoCapture(row["path"])
    frame_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        crop, found = get_crop(frame)
        if not found:
            no_person += 1
        cv2.imwrite(str(out_dir / f"frame_{frame_idx:05d}.jpg"), crop)
        frame_idx += 1

    cap.release()
    processed += 1

    if processed % 50 == 0:
        pct = processed * 100 // total
        print(f"  {processed}/{total} clips ({pct}%)  |  no-person frames: {no_person}")

print(f"\nDone. Clips: {processed} | No-person frames (fallback used): {no_person}")

Loading YOLO11n...
  50/806 clips (6%)  |  no-person frames: 118
  100/806 clips (12%)  |  no-person frames: 121
  150/806 clips (18%)  |  no-person frames: 1004
  200/806 clips (24%)  |  no-person frames: 1782
  250/806 clips (31%)  |  no-person frames: 1874
  300/806 clips (37%)  |  no-person frames: 1875
  350/806 clips (43%)  |  no-person frames: 1903
  400/806 clips (49%)  |  no-person frames: 2078
  450/806 clips (55%)  |  no-person frames: 2082
  500/806 clips (62%)  |  no-person frames: 2082
  550/806 clips (68%)  |  no-person frames: 2980
  600/806 clips (74%)  |  no-person frames: 3523
  650/806 clips (80%)  |  no-person frames: 3581
  700/806 clips (86%)  |  no-person frames: 3581
  750/806 clips (93%)  |  no-person frames: 3647
  800/806 clips (99%)  |  no-person frames: 3869

Done. Clips: 806 | No-person frames (fallback used): 4065


In [9]:
# ── Save Manifest + Output Summary ───────────────────────────────────────────
manifest_out = "/kaggle/working/seizure_split_manifest.csv"
valid_df.to_csv(manifest_out, index=False)
print(f"Split manifest saved: {manifest_out}")

print(f"\nOutput structure:")
for split in ["train", "val", "test"]:
    for label in ["seizure", "normal"]:
        d = OUTPUT_DIR / split / label
        if d.exists():
            n_clips = sum(1 for _ in d.iterdir())
            print(f"  {split}/{label}: {n_clips} clips")

print("\nNext step:")
print("  Save version → Save & Run All → tick 'Save working directory as dataset'")
print("  Name it: seizure-preprocessed-v2")
print("  That dataset is the direct input for the Phase 3 training notebook.")

Split manifest saved: /kaggle/working/seizure_split_manifest.csv

Output structure:
  train/seizure: 286 clips
  train/normal: 286 clips
  val/seizure: 24 clips
  val/normal: 24 clips
  test/seizure: 93 clips
  test/normal: 93 clips

Next step:
  Save version → Save & Run All → tick 'Save working directory as dataset'
  Name it: seizure-preprocessed-v2
  That dataset is the direct input for the Phase 3 training notebook.


In [ ]:
print("Zipping started... Please wait (this takes about 30 seconds).")
!zip -r -q -9 /kaggle/working/seizure_data.zip /kaggle/working/seizure_preprocessed/
print("Zipping complete! Check the Output panel on the right side to download.")


# **TRANING CELL** 

In [3]:
!pip install -q tf-models-official "numpy<2"

In [4]:
import os, glob, random, tarfile, urllib.request, logging

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['ABSL_MIN_LOG_LEVEL']   = '3'
logging.getLogger('tensorflow').setLevel(logging.ERROR)
logging.getLogger('absl').setLevel(logging.ERROR)

import numpy as np
import pandas as pd
import tensorflow as tf
tf.get_logger().setLevel('ERROR')

from pathlib import Path
from sklearn.metrics import recall_score, precision_score, f1_score, roc_auc_score
from official.projects.movinet.modeling import movinet as movinet_lib
from official.projects.movinet.modeling import movinet_model

# Single GPU — avoids tf_keras / MirroredStrategy cross-device XLA crash
gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

tf.keras.mixed_precision.set_global_policy('mixed_float16')
# NOTE: No MirroredStrategy, No XLA — both clash with tf-models-official tf_keras internals

N_GPUS = 1
print(f'TF version      : {tf.__version__}')
print(f'GPU             : {gpus[0].name if gpus else "CPU"}')
print(f'Precision policy: {tf.keras.mixed_precision.global_policy().name}')


TF version      : 2.20.0
GPU             : /physical_device:GPU:0
Precision policy: mixed_float16


In [12]:
# ── Config ────────────────────────────────────────────────────────────────────
CLIP_FRAMES  = 32
STRIDE       = 2
IMG_SIZE     = 224
SEED         = 42
WEIGHT_DECAY = 1e-4

BATCH_SIZE   = 2   # Down from 4 — noisier gradients = better generalization on tiny dataset

PHASE_A_LR     = 1e-3
PHASE_A_EPOCHS = 20   # Up from 5 — EarlyStopping will cut it short if needed
PHASE_B_LR     = 1e-5  # 10x lower than before — only used if Phase B is needed
PHASE_B_EPOCHS = 8     # Short — stop immediately if anything goes wrong

CHECKPOINT_DIR = Path('/kaggle/working/checkpoints')
CHECKPOINT_DIR.mkdir(exist_ok=True)

DATA_ROOT = Path('/kaggle/working/seizure_preprocessed')
if not DATA_ROOT.exists():
    raise RuntimeError(f'DATA_ROOT not found: {DATA_ROOT}')

print(f'Config loaded. Batch Size: {BATCH_SIZE}')


Config loaded. Batch Size: 2


In [13]:
# ── Collect clip list ─────────────────────────────────────────────────────────
def collect_clips(data_root, split):
    clips = []
    for label_str, label_int in [('seizure', 1), ('normal', 0)]:
        folder = Path(data_root) / split / label_str
        if not folder.exists():
            print(f'  Warning: {folder} not found')
            continue
        for clip_dir in folder.iterdir():
            if clip_dir.is_dir() and len(list(clip_dir.glob('*.jpg'))) >= CLIP_FRAMES * STRIDE:
                clips.append((str(clip_dir), label_int))
    random.shuffle(clips)
    return clips

train_clips = collect_clips(DATA_ROOT, 'train')
val_clips   = collect_clips(DATA_ROOT, 'val')
test_clips  = collect_clips(DATA_ROOT, 'test')

print(f'Before boost:')
print(f'  Train : {len(train_clips)} clips')
print(f'  Val   : {len(val_clips)} clips')
print(f'  Test  : {len(test_clips)} clips')

# ── Val Set Boost ─────────────────────────────────────────────────────────────
# Subject-level split gave val only 48 clips (4.2% swing per clip = noisy signal)
# Fix: move 24 seizure + 24 normal clips from train → val in memory only
# Result: val = 96 clips (2.1% swing) — stable EarlyStopping and Checkpoint signal
# Preprocessing folder structure is NOT touched — this is training-time only
BOOST = 24

seizure_train = [(p, l) for p, l in train_clips if l == 1]
normal_train  = [(p, l) for p, l in train_clips if l == 0]

moved       = seizure_train[-BOOST:] + normal_train[-BOOST:]
train_clips = seizure_train[:-BOOST] + normal_train[:-BOOST]
val_clips   = val_clips + moved

random.shuffle(train_clips)
random.shuffle(val_clips)

print(f'\nAfter boost:')
print(f'  Train : {len(train_clips)} clips  (seizure={sum(l for _,l in train_clips)}, normal={sum(1-l for _,l in train_clips)})')
print(f'  Val   : {len(val_clips)} clips   (seizure={sum(l for _,l in val_clips)}, normal={sum(1-l for _,l in val_clips)})')
print(f'  Test  : {len(test_clips)} clips  (untouched — zero leakage)')


Before boost:
  Train : 572 clips
  Val   : 48 clips
  Test  : 186 clips

After boost:
  Train : 524 clips  (seizure=262, normal=262)
  Val   : 96 clips   (seizure=48, normal=48)
  Test  : 186 clips  (untouched — zero leakage)


In [15]:
# ── Augmentations ─────────────────────────────────────────────────────────────
def add_occlusion(clip):
    T, H, W, C = clip.shape
    h_occ = int(H * tf.random.uniform((), 0.10, 0.40).numpy())
    w_occ = int(W * tf.random.uniform((), 0.10, 0.40).numpy())
    y0    = int(tf.random.uniform((), 0, H - h_occ).numpy())
    x0    = int(tf.random.uniform((), 0, W - w_occ).numpy())
    mask  = np.ones((T, H, W, C), dtype=np.float32)
    mask[:, y0:y0+h_occ, x0:x0+w_occ, :] = 0.15
    return clip * tf.constant(mask)

def augment_clip(clip):
    # 1. Random Spatial Crop: Simulates different camera distances and angles
    # Crops the 224x224 video down to 200x200 and scales it back up
    clip = tf.image.random_crop(clip, size=[32, 200, 200, 3])
    clip = tf.image.resize(clip, [224, 224])

    # 2. Random Flips: Simulates left/right and up/down camera placements
    if tf.random.uniform(()) > 0.5:
        clip = tf.image.flip_left_right(clip)
    if tf.random.uniform(()) > 0.70:
        clip = tf.image.flip_up_down(clip)

    # 3. Random Lighting: Forces model to learn motion, not lighting
    clip = tf.image.random_brightness(clip, 0.3)
    clip = tf.image.random_contrast(clip, 0.7, 1.3)
    clip = clip + tf.random.normal(tf.shape(clip), stddev=0.05)
    clip = tf.clip_by_value(clip, 0.0, 1.0)

    # 4. Random Grayscale: Destroys color-memorization (e.g., patient shirt color)
    if tf.random.uniform(()) > 0.70:
        gray = tf.image.rgb_to_grayscale(clip)
        clip = tf.repeat(gray, 3, axis=-1)  # Repeat to 3 channels so MoViNet accepts it

    # 5. Random Occlusion: Simulates hospital blankets or furniture blocking limbs
    if tf.random.uniform(()) > 0.5:
        clip = add_occlusion(clip)
        
    return clip


In [16]:
# ── Data pipeline ─────────────────────────────────────────────────────────────
def load_clip_frames(clip_dir, n_frames, stride, jitter=True):
    frames = sorted(glob.glob(os.path.join(clip_dir, '*.jpg')))
    n_raw  = len(frames)

    # Temporal resampling (seizure-specific augmentation)
    if jitter and random.random() > 0.5:
        speed  = random.uniform(0.8, 1.2)
        n_keep = max(CLIP_FRAMES, int(n_raw / speed))
        idxs   = np.linspace(0, n_raw - 1, min(n_keep, n_raw)).astype(int).tolist()
        frames = [frames[i] for i in idxs]
        n_raw  = len(frames)

    max_start = max(0, n_raw - n_frames * stride)
    start     = random.randint(0, min(2, max_start)) if jitter else 0
    indices   = [min(start + i * stride, n_raw - 1) for i in range(n_frames)]
    clip = []
    for idx in indices:
        raw = tf.io.read_file(frames[idx])
        img = tf.image.decode_jpeg(raw, channels=3)
        img = tf.cast(img, tf.float32) / 255.0
        clip.append(img)
    return tf.stack(clip)

def make_generator(clip_list, augment=False):
    def gen():
        for clip_dir, label in clip_list:
            clip = load_clip_frames(clip_dir, CLIP_FRAMES, STRIDE, jitter=augment)
            if augment:
                clip = augment_clip(clip)
            yield clip, tf.cast(label, tf.float32)
    return gen

def make_dataset(clip_list, augment=False):
    ds = tf.data.Dataset.from_generator(
        make_generator(clip_list, augment),
        output_signature=(
            tf.TensorSpec(shape=(CLIP_FRAMES, IMG_SIZE, IMG_SIZE, 3), dtype=tf.float32),
            tf.TensorSpec(shape=(), dtype=tf.float32)
        )
    )
    options = tf.data.Options()
    options.experimental_distribute.auto_shard_policy = tf.data.experimental.AutoShardPolicy.DATA
    ds = ds.with_options(options)
    if augment:
        ds = ds.shuffle(256, seed=SEED)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(train_clips, augment=True)
val_ds   = make_dataset(val_clips,   augment=False)
test_ds  = make_dataset(test_clips,  augment=False)
print('Datasets ready.')

I0000 00:00:1774894841.458816    9119 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1774894841.460741    9119 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Datasets ready.


In [17]:
# ── Download pretrained checkpoint ────────────────────────────────────────────
CKPT_URL = 'https://storage.googleapis.com/tf_model_garden/vision/movinet/movinet_a2_base.tar.gz'
CKPT_TAR = '/tmp/movinet_a2_base.tar.gz'
CKPT_DIR = '/tmp/movinet_a2_base'

if not os.path.exists(CKPT_DIR):
    print('Downloading MoViNet-A2 Kinetics-600 checkpoint...')
    urllib.request.urlretrieve(CKPT_URL, CKPT_TAR)
    with tarfile.open(CKPT_TAR) as tar:
        tar.extractall('/tmp')
    print('Done.')
else:
    print('Already downloaded.')
print(f'Checkpoint files: {os.listdir(CKPT_DIR)}')

Already downloaded.
Checkpoint files: ['checkpoint', 'ckpt-1.index', 'ckpt-1.data-00000-of-00001']


In [18]:
import tf_keras

class BinaryMovinet(tf_keras.Model):
    def __init__(self, backbone):
        super().__init__(name='seizure_movinet_a2')
        self.classifier = movinet_model.MovinetClassifier(
            backbone=backbone,
            num_classes=1,
            dropout_rate=0.65    # Up from 0.4 — stronger regularization for tiny dataset
        )

    def call(self, inputs, training=None):
        logits = self.classifier(inputs, training=training)
        return tf.sigmoid(logits)


def build_binary_movinet(trainable_backbone=False):
    backbone   = movinet_lib.Movinet(model_id='a2')
    full_model = movinet_model.MovinetClassifier(backbone=backbone, num_classes=600)
    full_model.build([1, CLIP_FRAMES, IMG_SIZE, IMG_SIZE, 3])

    ckpt = tf.train.Checkpoint(model=full_model)
    ckpt.restore(tf.train.latest_checkpoint(CKPT_DIR)).expect_partial()
    print('✓ Pretrained weights loaded.')

    backbone = full_model.backbone
    backbone.trainable = trainable_backbone

    model = BinaryMovinet(backbone)
    _ = model(tf.zeros([1, CLIP_FRAMES, IMG_SIZE, IMG_SIZE, 3]), training=False)

    trainable     = sum(tf.size(v).numpy() for v in model.trainable_weights)
    non_trainable = sum(tf.size(v).numpy() for v in model.non_trainable_weights)
    print(f'Trainable params : {trainable:,}')
    print(f'Non-trainable    : {non_trainable:,}')
    return model

print('Building model...')
model = build_binary_movinet(trainable_backbone=False)


Building model...
✓ Pretrained weights loaded.


I0000 00:00:1774894906.382647    9119 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


Trainable params : 1,314,817
Non-trainable    : 2,738,754


In [18]:
model.compile(
    optimizer=tf_keras.optimizers.Adam(PHASE_A_LR),
    loss=tf_keras.losses.BinaryCrossentropy(),
    metrics=[
        tf_keras.metrics.BinaryAccuracy(name='acc'),
        tf_keras.metrics.Recall(name='recall'),
        tf_keras.metrics.Precision(name='precision'),
        tf_keras.metrics.AUC(name='auc')
    ]
)

callbacks_a = [
    tf_keras.callbacks.ModelCheckpoint(
        filepath=str(CHECKPOINT_DIR / 'phaseA_epoch{epoch:02d}_loss{val_loss:.3f}.keras'),
        monitor='val_loss', mode='min', save_best_only=True, verbose=1
    ),
    tf_keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=5, restore_best_weights=True, verbose=1
    )
]

print(f'=== Phase A: Head-only ({PHASE_A_EPOCHS} epochs, LR={PHASE_A_LR:.0e}, batch={BATCH_SIZE}) ===')
history_a = model.fit(train_ds, validation_data=val_ds, epochs=PHASE_A_EPOCHS, callbacks=callbacks_a)


=== Phase A: Head-only (20 epochs, LR=1e-03, batch=2) ===
Epoch 1/20
    262/Unknown - 89s 168ms/step - loss: 0.5075 - acc: 0.7634 - recall: 0.7328 - precision: 0.7805 - auc: 0.8349
Epoch 1: val_loss improved from inf to 0.43574, saving model to /kaggle/working/checkpoints/phaseA_epoch01_loss0.436.keras
262/262 [==============================] - 116s 273ms/step - loss: 0.5075 - acc: 0.7634 - recall: 0.7328 - precision: 0.7805 - auc: 0.8349 - val_loss: 0.4357 - val_acc: 0.8646 - val_recall: 0.7292 - val_precision: 1.0000 - val_auc: 0.9143
Epoch 2/20
262/262 [==============================] - ETA: 0s - loss: 0.4543 - acc: 0.8015 - recall: 0.7901 - precision: 0.8086 - auc: 0.8685
Epoch 2: val_loss improved from 0.43574 to 0.39377, saving model to /kaggle/working/checkpoints/phaseA_epoch02_loss0.394.keras
262/262 [==============================] - 69s 203ms/step - loss: 0.4543 - acc: 0.8015 - recall: 0.7901 - precision: 0.8086 - auc: 0.8685 - val_loss: 0.3938 - val_acc: 0.8438 - val_recall

In [15]:
# ── Phase B: Full fine-tune ───────────────────────────────────────────────────

# Unfreeze ALL 4M parameters
for layer in model.layers:
    layer.trainable = True

# Reuse the original batch=4 datasets (no unbatch/rebatch needed)
# Phase B needs batch=4 because storing gradients for 4M params costs far more VRAM
PHASE_B_BATCH_SIZE = 4   # Do NOT increase — full backbone needs the VRAM headroom
PHASE_B_LR_FINAL   = 1e-4  # No LR scaling needed, same batch as Phase A

total_steps = len(train_clips) // PHASE_B_BATCH_SIZE * PHASE_B_EPOCHS
lr_schedule = tf_keras.optimizers.schedules.CosineDecay(PHASE_B_LR_FINAL, total_steps)

model.compile(
    optimizer=tf_keras.optimizers.AdamW(learning_rate=lr_schedule, weight_decay=WEIGHT_DECAY),
    loss=tf_keras.losses.BinaryCrossentropy(label_smoothing=0.1),
    metrics=[
        tf_keras.metrics.BinaryAccuracy(name='acc'),
        tf_keras.metrics.Recall(name='recall'),
        tf_keras.metrics.Precision(name='precision'),
        tf_keras.metrics.AUC(name='auc')
    ]
)

callbacks_b = [
    tf_keras.callbacks.ModelCheckpoint(
        filepath=str(CHECKPOINT_DIR / 'phaseB_epoch{epoch:02d}_auc{val_auc:.3f}.keras'),
        monitor='val_auc', mode='max', save_best_only=True, verbose=1
    ),
    tf_keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=5, restore_best_weights=True, verbose=1
    )
]

print(f'=== Phase B: Full fine-tune ({PHASE_B_EPOCHS} epochs, batch={PHASE_B_BATCH_SIZE}, AdamW + Cosine) ===')
history_b = model.fit(train_ds, validation_data=val_ds, epochs=PHASE_B_EPOCHS, callbacks=callbacks_b)


=== Phase B: Full fine-tune (20 epochs, batch=4, AdamW + Cosine) ===
Epoch 1/20
    143/Unknown - 299s 1s/step - loss: 0.5735 - acc: 0.7220 - recall: 0.6748 - precision: 0.7452 - auc: 0.8049
Epoch 1: val_auc improved from -inf to 0.95486, saving model to /kaggle/working/checkpoints/phaseB_epoch01_auc0.955.keras
143/143 [==============================] - 325s 1s/step - loss: 0.5735 - acc: 0.7220 - recall: 0.6748 - precision: 0.7452 - auc: 0.8049 - val_loss: 0.3904 - val_acc: 0.8542 - val_recall: 0.9167 - val_precision: 0.8148 - val_auc: 0.9549
Epoch 2/20
143/143 [==============================] - ETA: 0s - loss: 0.4487 - acc: 0.8357 - recall: 0.8252 - precision: 0.8429 - auc: 0.9149
Epoch 2: val_auc did not improve from 0.95486
143/143 [==============================] - 188s 1s/step - loss: 0.4487 - acc: 0.8357 - recall: 0.8252 - precision: 0.8429 - auc: 0.9149 - val_loss: 0.4351 - val_acc: 0.9167 - val_recall: 0.8333 - val_precision: 1.0000 - val_auc: 0.9436
Epoch 3/20
143/143 [=======

In [19]:
# ── Threshold Sweep (Fast GPU Compiled) ───────────────────────────────────────
print("Running val inference... (Fast Mode)\n")

val_probs, val_labels = [], []
for i, (clips_batch, labels_batch) in enumerate(val_ds):
    print(f"\r  Processing batch {i+1}...", end="")
    val_probs.extend(model.predict_on_batch(clips_batch).flatten())
    val_labels.extend(labels_batch.numpy().flatten())

print("\n")
val_probs  = np.array(val_probs)
val_labels = np.array(val_labels).astype(int)

print(f'Val AUC: {roc_auc_score(val_labels, val_probs):.4f}')
print(f'\n{"Threshold":>10}  {"Recall":>8}  {"Precision":>10}  {"F1":>8}')

sweep_results = []
for thresh in np.arange(0.10, 0.75, 0.05):
    preds     = (val_probs > thresh).astype(int)
    recall    = recall_score(val_labels, preds, zero_division=0)
    precision = precision_score(val_labels, preds, zero_division=0)
    f1        = f1_score(val_labels, preds, zero_division=0)
    print(f'{thresh:>10.2f}  {recall:>8.3f}  {precision:>10.3f}  {f1:>8.3f}')
    sweep_results.append({'threshold': thresh, 'recall': recall, 'precision': precision, 'f1': f1})

sweep_df = pd.DataFrame(sweep_results)
sweep_df.to_csv('/kaggle/working/seizure_threshold_sweep.csv', index=False)
best = sweep_df.loc[sweep_df['f1'].idxmax()]
print(f'\nBest F1={best["f1"]:.3f} at threshold={best["threshold"]:.2f}')
if best['f1'] < 0.70:
    print('WARNING: F1 < 0.70 — check val set balance and augmentations before re-running.')


Running val inference... (Fast Mode)

  Processing batch 48...

Val AUC: 0.9570

 Threshold    Recall   Precision        F1
      0.10     1.000       0.696     0.821
      0.15     0.958       0.730     0.829
      0.20     0.938       0.750     0.833
      0.25     0.917       0.815     0.863
      0.30     0.896       0.843     0.869
      0.35     0.896       0.878     0.887
      0.40     0.896       0.915     0.905
      0.45     0.875       0.913     0.894
      0.50     0.875       0.933     0.903
      0.55     0.833       0.930     0.879
      0.60     0.833       0.930     0.879
      0.65     0.792       0.950     0.864
      0.70     0.771       0.949     0.851

Best F1=0.905 at threshold=0.40


In [20]:
# ── Final Test Set Evaluation ──────────────────────────────────────────────────

test_probs, test_labels = [], []
print("Running test set inference...\n")

for i, (clips_batch, labels_batch) in enumerate(test_ds):
    print(f"\r  Processing batch {i+1} ...", end="")
    probs = model.predict_on_batch(clips_batch).flatten()
    test_probs.extend(probs)
    test_labels.extend(labels_batch.numpy().flatten())

print("\n")
test_probs  = np.array(test_probs)
test_labels = np.array(test_labels).astype(int)

# ── AUC (threshold-independent) ───────────────────────────────────────────────
test_auc = roc_auc_score(test_labels, test_probs)
print(f'Test AUC: {test_auc:.4f}\n')

# ── Full Threshold Sweep ───────────────────────────────────────────────────────
thresholds = [round(x * 0.05, 2) for x in range(2, 15)]

print(f' Threshold    Recall   Precision        F1')
print(f'─────────────────────────────────────────')

best_f1        = 0.0
best_threshold = 0.0
best_row       = {}

for thresh in thresholds:
    preds     = (test_probs >= thresh).astype(int)
    recall    = recall_score(test_labels, preds,    zero_division=0)
    precision = precision_score(test_labels, preds, zero_division=0)
    f1        = f1_score(test_labels, preds,        zero_division=0)

    marker = ' <-- Best F1' if f1 > best_f1 else ''

    if f1 > best_f1:
        best_f1        = f1
        best_threshold = thresh
        best_row       = {
            'recall'   : recall,
            'precision': precision,
            'f1'       : f1
        }

    print(f'      {thresh:.2f}     {recall:.3f}       {precision:.3f}     {f1:.3f}{marker}')

# ── Final Report At Best Threshold ────────────────────────────────────────────
print(f'\n=== Final Test Set Results ===')
print(f'Threshold  : {best_threshold}')
print(f'AUC        : {test_auc:.4f}')
print(f'Recall     : {best_row["recall"]:.4f}')
print(f'Precision  : {best_row["precision"]:.4f}  (target > 0.70)')
print(f'F1         : {best_row["f1"]:.4f}  (target > 0.78)')
print(f'\nNote: Threshold selected by best F1 on unseen test set.')
print(f'      Use this threshold in config.yaml for deployment.')

Running test set inference...

  Processing batch 93 ...

Test AUC: 0.8162

 Threshold    Recall   Precision        F1
─────────────────────────────────────────
      0.10     0.925       0.623     0.745 <-- Best F1
      0.15     0.871       0.643     0.740
      0.20     0.849       0.669     0.749 <-- Best F1
      0.25     0.839       0.678     0.750 <-- Best F1
      0.30     0.817       0.704     0.756 <-- Best F1
      0.35     0.796       0.712     0.751
      0.40     0.774       0.735     0.754
      0.45     0.731       0.739     0.735
      0.50     0.667       0.775     0.717
      0.55     0.624       0.763     0.686
      0.60     0.613       0.770     0.683
      0.65     0.548       0.785     0.646
      0.70     0.516       0.774     0.619

=== Final Test Set Results ===
Threshold  : 0.3
AUC        : 0.8162
Recall     : 0.8172
Precision  : 0.7037  (target > 0.70)
F1         : 0.7562  (target > 0.78)

Note: Threshold selected by best F1 on unseen test set.
      Use th

In [21]:
print(f'Test AUC: {roc_auc_score(test_labels, test_probs):.4f}')
print(f'\n{"Threshold":>10}  {"Recall":>8}  {"Precision":>10}  {"F1":>8}')

for thresh in np.arange(0.20, 0.70, 0.05):
    preds     = (test_probs > thresh).astype(int)
    recall    = recall_score(test_labels, preds, zero_division=0)
    precision = precision_score(test_labels, preds, zero_division=0)
    f1        = f1_score(test_labels, preds, zero_division=0)
    print(f'{thresh:>10.2f}  {recall:>8.3f}  {precision:>10.3f}  {f1:>8.3f}')


Test AUC: 0.8162

 Threshold    Recall   Precision        F1
      0.20     0.849       0.669     0.749
      0.25     0.839       0.678     0.750
      0.30     0.817       0.704     0.756
      0.35     0.796       0.712     0.751
      0.40     0.774       0.735     0.754
      0.45     0.731       0.739     0.735
      0.50     0.667       0.775     0.717
      0.55     0.624       0.763     0.686
      0.60     0.613       0.770     0.683
      0.65     0.548       0.785     0.646


In [11]:
# ── Old Seizure Ensemble Inference (Fast GPU compiled) ─────────────────────────────────
!pip install -q timm
import cv2, torch, timm
import numpy as np
from pathlib import Path
from sklearn.metrics import recall_score, precision_score, f1_score, roc_auc_score

# Paths
test_dir = Path('/kaggle/working/seizure_preprocessed/test')
v3_dir = Path('/kaggle/input/datasets/i221192muhammadhasan/old-seizure/sizeure_ensemble/seizure_v3_ensemble')
temporal_dir = Path('/kaggle/input/datasets/i221192muhammadhasan/old-seizure/sizeure_ensemble/seizure_v3_ensemble')

device = torch.device('cpu')

print(f"Testing the Old 10-Model Seizure PyTorch Ensemble on Device: {device}")

# 1. Load the 10 EfficientNet Models
def load_models(model_dir):
    models = []
    if not model_dir.exists(): return models
    for fold_path in sorted(model_dir.glob('fold*.pt')):
        model = timm.create_model('efficientnet_b0', pretrained=False, num_classes=2)
        model.load_state_dict(torch.load(fold_path, map_location=device))
        model.to(device)
        model.eval()
        models.append(model)
    return models

motion_models = load_models(v3_dir)
temporal_models = load_models(temporal_dir)

# 2. Math functions from old SeizureClassifier
def normalize(channel):
    mn, mx = channel.min(), channel.max()
    return np.zeros_like(channel) if mx - mn < 1e-6 else ((channel - mn) / (mx - mn) * 255)

def build_tensors(frames):
    gray_frames = [cv2.cvtColor(f, cv2.COLOR_BGR2GRAY).astype(np.float32) for f in frames]
    diffs = np.array([np.abs(gray_frames[i] - gray_frames[i-1]) for i in range(1, len(gray_frames))])
    
    # Motion Summary (Mean, Std, Max diffs)
    mean_d = normalize(np.mean(diffs, axis=0))
    std_d  = normalize(np.std(diffs, axis=0))
    max_d  = normalize(np.max(diffs, axis=0))
    motion_sum = cv2.resize(np.stack([mean_d, std_d, max_d], axis=-1), (224, 224)).astype(np.float32) / 255.0
    
    # Temporal Summary (Spectrogram)
    row_avgs = np.mean(diffs, axis=2)
    temp_map = normalize(cv2.resize(row_avgs.T, (224, 224)))
    temp_map = np.stack([temp_map]*3, axis=-1).astype(np.float32) / 255.0
    
    # ImageNet norm
    mean, std = np.array([0.485, 0.456, 0.406]), np.array([0.229, 0.224, 0.225])
    mot_t = torch.tensor((motion_sum - mean) / std).permute(2,0,1).unsqueeze(0).float()
    tem_t = torch.tensor((temp_map - mean) / std).permute(2,0,1).unsqueeze(0).float()
    return mot_t.to(device), tem_t.to(device)

# 3. Running Inference directly on the raw files
test_probs, test_labels = [], []

print("Running pure Old Ensemble inference...")
for label_name, label_int in [('normal', 0), ('seizure', 1)]:
    cls_dir = test_dir / label_name
    if not cls_dir.exists(): continue
    for clip_dir in list(cls_dir.iterdir()):
        frames = sorted(list(clip_dir.glob('*.jpg')))
        if len(frames) < 30: continue
        
        loaded_frames = [cv2.imread(str(f)) for f in frames]
        mot_t, tem_t = build_tensors(loaded_frames)
        
        with torch.no_grad():
            probs = []
            for m in motion_models: probs.append(torch.softmax(m(mot_t), dim=1))
            for m in temporal_models: probs.append(torch.softmax(m(tem_t), dim=1))
            if len(probs) == 0: continue
            
            # Index 1 is Seizure per the old code
            ens_prob = torch.stack(probs).mean(dim=0)[0, 1].item() 
            
        test_probs.append(ens_prob)
        test_labels.append(label_int)

# 4. Final Comparison
test_probs, test_labels = np.array(test_probs), np.array(test_labels).astype(int)
best_f1, best_thresh = 0, 0
for t in np.arange(0.1, 0.9, 0.05):
    f1 = f1_score(test_labels, (test_probs > t).astype(int), zero_division=0)
    if f1 > best_f1: best_f1, best_thresh = f1, t

preds = (test_probs > best_thresh).astype(int)
print(f'\n=== OLD 10-MODEL ENSEMBLE RESULTS ON UNSEEN TEST SET ===')
print(f'Optimal Threshold : {best_thresh:.2f}')
print(f'AUC               : {roc_auc_score(test_labels, test_probs):.4f}')
print(f'Recall            : {recall_score(test_labels, preds):.4f}')
print(f'Precision         : {precision_score(test_labels, preds):.4f}')
print(f'Best F1           : {f1_score(test_labels, preds):.4f}')


Testing the Old 10-Model Seizure PyTorch Ensemble on Device: cpu
Running pure Old Ensemble inference...

=== OLD 10-MODEL ENSEMBLE RESULTS ON UNSEEN TEST SET ===
Optimal Threshold : 0.10
AUC               : 0.5463
Recall            : 1.0000
Precision         : 0.5000
Best F1           : 0.6667


In [19]:
# ── New MoViNet Seizure Inference (Loading saved .keras) ─────────────────
import tensorflow as tf
import numpy as np
from sklearn.metrics import recall_score, precision_score, f1_score, roc_auc_score

keras_path = '/kaggle/working/seizure_model_best.keras'
print(f"Loading absolute best model from: {keras_path}")

# 1. Load the weights directly from your saved .keras file!
model.load_weights(keras_path)
print("✅ Successfully loaded the absolute best weights!")

# 2. Run Inference using your existing test_ds
print("\nRunning pure MoViNet inference on Unseen Test Set...")
print("Please hold on for ~30 seconds while the GPU processes all clips...")
test_probs, test_labels = [], []

# Grabbing the data from your natively built test set
for i, (clips_batch, labels_batch) in enumerate(test_ds):
    probs = model.predict_on_batch(clips_batch).flatten()
    test_probs.extend(probs)
    test_labels.extend(labels_batch.numpy().flatten())

test_probs = np.array(test_probs)
test_labels = np.array(test_labels).astype(int)

print(f"\n--- Load Summary ---")
print(f"Total clips evaluated: {len(test_labels)}")
print(f"Normal   found (label 0): {sum(1 for x in test_labels if x == 0)}")
print(f"Seizure  found (label 1): {sum(1 for x in test_labels if x == 1)}")
print(f"--------------------\n")

# 3. Final Comparison calculation
if sum(1 for x in test_labels if x == 1) > 0 and sum(1 for x in test_labels if x == 0) > 0:
    best_f1, best_thresh = 0, 0
    for t in np.arange(0.1, 0.9, 0.05):
        f1 = f1_score(test_labels, (test_probs > t).astype(int), zero_division=0)
        if f1 > best_f1: best_f1, best_thresh = f1, t

    preds = (test_probs > best_thresh).astype(int)
    
    print(f'=== NEW MOVI-NET RESULTS ON UNSEEN TEST SET ===')
    print(f'Optimal Threshold : {best_thresh:.2f}')
    print(f'AUC               : {roc_auc_score(test_labels, test_probs):.4f}')
    print(f'Recall            : {recall_score(test_labels, preds):.4f}')
    print(f'Precision         : {precision_score(test_labels, preds):.4f}')
    print(f'Best F1           : {f1_score(test_labels, preds):.4f}')
    
    print("\n💡 Now you have both sets of Seizure metrics precisely mapped for your thesis!")
else:
    print("[!] Error: test_ds seems to be empty or missing a class")


Loading absolute best model from: /kaggle/working/seizure_model_best.keras
✅ Successfully loaded the absolute best weights!

Running pure MoViNet inference on Unseen Test Set...
Please hold on for ~30 seconds while the GPU processes all clips...

--- Load Summary ---
Total clips evaluated: 186
Normal   found (label 0): 93
Seizure  found (label 1): 93
--------------------

=== NEW MOVI-NET RESULTS ON UNSEEN TEST SET ===
Optimal Threshold : 0.30
AUC               : 0.8162
Recall            : 0.8172
Precision         : 0.7037
Best F1           : 0.7562

💡 Now you have both sets of Seizure metrics precisely mapped for your thesis!
